# 🥔 Potato Chip Defect Detection — CNN Model

This notebook builds a Convolutional Neural Network (CNN) to classify potato chip images as **defective** or **non-defective**.

**Pipeline:**
1. Download dataset via KaggleHub
2. Load & augment images with ImageDataGenerator
3. Build and train a CNN
4. Evaluate with confusion matrix & classification report
5. Save trained model to `models/cnn_model.h5`

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [1]:
import os
print("os done")

import numpy as np
print("numpy done")

import matplotlib.pyplot as plt
print("matplotlib done")



import kagglehub
print("kagglehub done")

import tensorflow as tf
print("tensorflow done")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
print("keras preprocessing done")

from tensorflow.keras.models import Sequential
print("Sequential done")

from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
print("layers done")

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
print("callbacks done")

from sklearn.metrics import confusion_matrix, classification_report
print("sklearn done")

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

os done
numpy done
matplotlib done


/Users/priyanshusoni/Desktop/Python Project/potato-chip-ai/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


kagglehub done


: 

## 2. Download Dataset

Using `kagglehub` to download the PepsiCo Lab Potato Chips Quality Control dataset.
The dataset contains ~1,000 images split into `defective` and `non-defective` folders.

In [ ]:
# Download the dataset
dataset_path = kagglehub.dataset_download("concaption/pepsico-lab-potato-quality-control")
print(f"Dataset downloaded to: {dataset_path}")

# List directory contents to find the image folders
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Only show 2 levels deep
        sub_indent = ' ' * 2 * (level + 1)
        for d in dirs:
            print(f"{sub_indent}{d}/")
        if files:
            print(f"{sub_indent}({len(files)} files)")

In [ ]:
# Detect the actual data directory
# The dataset may have nested folders — we need the parent that contains
# the class subdirectories (defective / non-defective)

data_dir = None

for root, dirs, files in os.walk(dataset_path):
    # Look for a directory that has subdirectories matching class names
    lower_dirs = [d.lower() for d in dirs]
    if any('defect' in d for d in lower_dirs):
        data_dir = root
        break

if data_dir is None:
    # Fallback: use the dataset_path itself
    data_dir = dataset_path

print(f"Using data directory: {data_dir}")
print(f"Subdirectories: {os.listdir(data_dir)}")

# Count images per class
for class_dir in sorted(os.listdir(data_dir)):
    class_path = os.path.join(data_dir, class_dir)
    if os.path.isdir(class_path):
        n_images = len([f for f in os.listdir(class_path) 
                       if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
        print(f"  {class_dir}: {n_images} images")

## 3. Data Loading & Augmentation

We use `ImageDataGenerator` to:
- **Rescale** pixel values to [0, 1]
- **Augment** training data with rotation, flips, zoom, and shear
- **Split** 80% training / 20% validation

In [ ]:
# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# Aggressive Training data generator to simulate real-world webcams and phone screens
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.5, 1.5], # Critical for glowing phone screens vs dark backgrounds
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(data_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='training', seed=SEED, shuffle=True)
val_generator = val_datagen.flow_from_directory(data_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='validation', seed=SEED, shuffle=False)

print(f'\nTraining samples: {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')
print(f'Class indices: {train_generator.class_indices}')


In [ ]:
# Visualize sample training images
class_names = list(train_generator.class_indices.keys())

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Training Images', fontsize=16, fontweight='bold')

images, labels = next(train_generator)
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        ax.set_title(class_names[int(labels[i])], fontsize=11)
        ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Build the CNN Model

Architecture:
```
Conv2D(32, 3×3, relu) → MaxPooling2D(2×2)
Conv2D(64, 3×3, relu) → MaxPooling2D(2×2)
Flatten → Dense(128, relu) → Dropout(0.5)
Dense(1, sigmoid)  — binary output
```

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Load Pre-trained MobileNetV2 (Feature Extractor)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model so we don't destroy the ImageNet features
base_model.trainable = False

# Create our custom top layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.summary()


In [ ]:
# Compile the model
# - Adam optimizer: adaptive learning rate, works well out of the box
# - Binary crossentropy: standard loss for binary classification
# - Accuracy: easy to interpret metric

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

## 5. Train the Model

We train for 20 epochs with:
- **ModelCheckpoint**: saves the best model (by validation accuracy)
- **EarlyStopping**: stops training if validation loss doesn't improve for 5 epochs

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Callbacks
checkpoint = ModelCheckpoint(
    '../models/cnn_model.h5',   # Save path
    monitor='val_accuracy',      # Watch validation accuracy
    save_best_only=True,         # Only save when val_accuracy improves
    mode='max',                  # Higher accuracy = better
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',          # Watch validation loss
    patience=5,                  # Stop after 5 epochs with no improvement
    restore_best_weights=True,   # Revert to best weights
    verbose=1
)

# Train the model
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=[checkpoint, early_stop],
    verbose=1
)

print("\n✅ Training complete!")

## 6. Training History — Accuracy & Loss Curves

These plots help diagnose overfitting (training accuracy keeps rising while validation plateaus or drops).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluation — Confusion Matrix & Classification Report

We evaluate on the full validation set to get:
- **Confusion matrix**: shows true positives, false positives, etc.
- **Classification report**: precision, recall, F1-score per class

In [ ]:
# Get predictions on validation set
val_generator.reset()
y_pred_proba = model.predict(val_generator, verbose=1)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
y_true = val_generator.classes

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            annot_kws={'size': 16})
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

# Classification Report
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Final evaluation metrics
val_loss, val_accuracy = model.evaluate(val_generator, verbose=0)

print(f"\n{'='*50}")
print(f"FINAL MODEL PERFORMANCE")
print(f"{'='*50}")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.1f}%)")
print(f"\n✅ Best model saved to: ../models/cnn_model.h5")

## 8. Test with Sample Images

Quick visual check — let's see what the model predicts on a few validation images.

In [ ]:
# Visualize predictions on sample validation images
val_generator.reset()
images, labels = next(val_generator)
predictions = model.predict(images, verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Model Predictions on Validation Images', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        pred_class = class_names[int(predictions[i] > 0.5)]
        true_class = class_names[int(labels[i])]
        confidence = predictions[i][0] if predictions[i] > 0.5 else 1 - predictions[i][0]
        
        color = 'green' if pred_class == true_class else 'red'
        ax.set_title(f"Pred: {pred_class}\nTrue: {true_class}\nConf: {confidence:.1%}",
                    fontsize=9, color=color, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

print("\n🎉 CNN model training complete! Model saved to models/cnn_model.h5")